In [1]:
import os
import numpy as np
import scipy.io as sio
from tqdm import tqdm
import h5py

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt

import sys
sys.path.append("..") 

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print(device)

cuda


In [2]:
from UCI.pre_processing import  load_UCI_dataset, create_dataloaders

In [ ]:
X_train, y_train, X_val, y_val, X_test, y_test = load_UCI_dataset(WINDOW_SIZE= 1000, STEP_SIZE= 500)

Total recordings: 12000
Train recordings: 8640
Validation recordings: 960
Test recordings: 2400


100%|██████████| 8640/8640 [01:36<00:00, 89.23it/s] 


Skipped recordings: 4


 34%|███▍      | 324/960 [00:03<00:07, 80.23it/s] 

In [ ]:
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

# The shape should be : (batch, channels, length), length = 8*125 = 1000samples, only one channel as PPG and 

In [ ]:
print(X_train.dtype)
print(y_train.dtype)

print(torch.isnan(X_train).any())
print(torch.isnan(y_train).any())

In [ ]:
cat ConvTran/Models/model.py

In [ ]:
train_loader, val_loader, test_loader = create_dataloaders(
    X_train,
    y_train,
    X_val,
    y_val,
    X_test,
    y_test,
    batch_size=64
)

X_batch, y_batch = next(iter(train_loader))

print("X:", X_batch.shape, X_batch.dtype)
print("y:", y_batch.shape, y_batch.dtype)

In [ ]:
config = {
    # Input
    'Data_shape': (32, 1, 1000),

    # ConvTran architecture
    'emb_size': 16,
    'num_heads': 8,
    'dim_ff': 256,

    # Positional encoding
    'Fix_pos_encode': 'tAPE',
    'Rel_pos_encode': 'eRPE',

    # Dropout
    'dropout': 0.01,

    # Regression
    'output_size': 2,

    # Model type
    'Net_Type': ['C-T'],
}

In [ ]:
import sys

sys.path.insert(
    0,
    "/data1/yashvi_bhuva/BP_estimation_using_PPG/ConvTran/ConvTran"
)
from Models.model import model_factory
model = model_factory(config)

X_batch, y_batch = next(iter(train_loader))

output = model(X_batch)

print("Input :", X_batch.shape)
print("Output:", output.shape)

In [ ]:
import torch.nn as nn

criterion = nn.SmoothL1Loss()

loss = criterion(output, y_batch)

print("Prediction shape:", output.shape)
print("Target shape:", y_batch.shape)
print("Loss:", loss.item())

loss.backward()

print("Backward pass successful")

In [ ]:
from torch.optim import AdamW

optimizer = AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

optimizer.step()

print("Optimizer step successful")

In [ ]:
model.train()

X_batch, y_batch = next(iter(train_loader))

optimizer.zero_grad()

pred = model(X_batch)

loss = criterion(pred, y_batch)

loss.backward()

optimizer.step()

print("Loss:", loss.item())
print("Training step successful")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model_factory(config).to(device)

In [ ]:
loss_module = torch.nn.SmoothL1Loss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

In [ ]:

from Training import SupervisedTrainer, validate, train_runner
trainer = SupervisedTrainer(
    model=model,
    dataloader=train_loader,
    device=device,
    loss_module=loss_module,
    optimizer=optimizer,
    l2_reg=None
)

val_evaluator = SupervisedTrainer(
    model=model,
    dataloader=val_loader,
    device=device,
    loss_module=loss_module,
    optimizer=None,
    l2_reg=None
)

In [ ]:
import os

config['epochs'] = 5
config['optimizer'] = optimizer
config['loss_module'] = loss_module
config['key_metric'] = 'loss'
config['save_dir'] = './checkpoints'

os.makedirs(config['save_dir'], exist_ok=True)

In [ ]:
# metrics = trainer.train_epoch(1)

# print(metrics)

In [ ]:
# val_metrics, results = val_evaluator.evaluate(1)

# print(val_metrics)

In [ ]:
config['epochs'] = 5

train_runner(
    config=config,
    model=model,
    trainer=trainer,
    val_evaluator=val_evaluator,
    path='./checkpoints/best_model.pth'
)

KeyboardInterrupt: 

In [ ]:
checkpoint = torch.load(
    './checkpoints/model_best.pth',
    map_location=device
)

model.load_state_dict(checkpoint['model_state_dict'])
model.to(device)

In [ ]:
test_evaluator = SupervisedTrainer(
    model=model,
    dataloader=test_loader,
    device=device,
    loss_module=loss_module,
    optimizer=None,
    l2_reg=None,
    print_interval=10,
    console=True,
    print_conf_mat=False
)

In [ ]:
test_metrics, test_results = test_evaluator.evaluate(
    epoch_num=None,
    keep_all=True
)

In [ ]:
print("Test Results:")
for key, value in test_metrics.items():
    print(f"{key}: {value:.4f}")

In [ ]:
y_true = test_results['targets']
y_pred = test_results['predictions']

print("True shape:", y_true.shape)
print("Pred shape:", y_pred.shape)

In [ ]:
import numpy as np
sbp_true = y_true[:, 0]
sbp_pred = y_pred[:, 0]

dbp_true = y_true[:, 1]
dbp_pred = y_pred[:, 1]

sbp_error = sbp_pred - sbp_true
dbp_error = dbp_pred - dbp_true

print("\n===== TEST RESULTS =====")

print(f"SBP MAE  : {np.mean(np.abs(sbp_error)):.2f} mmHg")
print(f"SBP RMSE : {np.sqrt(np.mean(sbp_error**2)):.2f} mmHg")
print(f"SBP ME   : {np.mean(sbp_error):.2f} mmHg")
print(f"SBP STD  : {np.std(sbp_error):.2f} mmHg")

print(f"\nDBP MAE  : {np.mean(np.abs(dbp_error)):.2f} mmHg")
print(f"DBP RMSE : {np.sqrt(np.mean(dbp_error**2)):.2f} mmHg")
print(f"DBP ME   : {np.mean(dbp_error):.2f} mmHg")
print(f"DBP STD  : {np.std(dbp_error):.2f} mmHg")